In [1]:
import duckdb

In [2]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [10]:
df = con.execute("""
SELECT * FROM (
SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) as row 
FROM bronze_z0019 
WHERE data_ingestao >= '2026-07-22')
   WHERE row = 1""").fetchdf()
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-22 19:29:38.019500,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-22 19:29:38.019500,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-22 19:34:44.059165,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-22 19:34:44.059165,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-22 19:34:44.059165,1


In [11]:
df.head(10)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-07-22 19:29:38.019500,1
1,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-07-22 19:29:38.019500,1
2,10004,SERRA,BT50,100,200,z0019_2.csv,2026-07-22 19:34:44.059165,1
3,10005,MACHADO,BT50,100,100,z0019_2.csv,2026-07-22 19:34:44.059165,1
4,10003,PREGO,BT10,100,60,z0019_2.csv,2026-07-22 19:34:44.059165,1


In [15]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={'NATBR': 'id',
                                    'MAKTX': 'nm_produto',
                                    'WERKS': 'id_categoria',
                                    'MAINS': 'id_fornecedor',
                                    'LABST': 'vl_preco'})
display(df_final)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10001,PARAFUSO,BT10,100,100
1,10002,MARTELO,BT50,100,1500
2,10004,SERRA,BT50,100,200
3,10005,MACHADO,BT50,100,100
4,10003,PREGO,BT10,100,60


In [17]:
df2 = df_final
df2 = df2.astype({
            'id': int,
            'nm_produto': str,
            'id_categoria': str,
            'id_fornecedor': int,
            'vl_preco': float
})

df2.dtypes

id                 int64
nm_produto        object
id_categoria      object
id_fornecedor      int64
vl_preco         float64
dtype: object

In [19]:
con.execute(""" 
    CREATE TABLE IF NOT EXISTS produtos (
        id BIGINT,
        nm_produto TEXT,
        id_categoria TEXT,
        id_fornecedor BIGINT,
        vlr_preco FLOAT    
    )

""")

In [23]:
df_resultado = con.execute("SELECT * FROM produtos").fetchdf()
display(df_resultado)

,id,nm_produto,id_categoria,id_fornecedor,vlr_preco
0,10001,PARAFUSO,BT10,100,100.0
1,10002,MARTELO,BT50,100,1500.0
2,10004,SERRA,BT50,100,200.0
3,10005,MACHADO,BT50,100,100.0
4,10003,PREGO,BT10,100,60.0


In [21]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [24]:
con.close()